# Intermediate 03 - Authentication

In this notebook:
- Learn the basics of SIP authentication
- Understand the Digest authentication flow
- Practice with `www_authorize` and `proxy_authorize`
- Handle authentication failure responses

---

## What is SIP Authentication?

SIP authentication uses the same mechanism as **HTTP Digest authentication**:

```
1. UA → Proxy: INVITE (without auth)
2. Proxy → UA: 407 Proxy Authentication Required
3. UA → Proxy: INVITE (with credentials)
4. Proxy → UA: Auth success → process request
```

Two authentication types:
- **WWW-Authenticate**: Registrar challenges UA (`401 Unauthorized`)
- **Proxy-Authenticate**: Proxy challenges UA (`407 Proxy Auth Required`)

In practice, **Proxy authentication (407)** is used most often.

## 1. Authentication for REGISTER

The most common authentication scenario. Verify credentials during REGISTER.

In [ ]:
%%sip REGISTER
From: <sip:1001@example.com>;tag=reg1
To: <sip:1001@example.com>

In [ ]:
# Unauthenticated REGISTER — reject with 401
$var(has_auth) = "no";

if ($var(has_auth) == "no") {
    xlog("No credentials — challenging REGISTER");
    send_reply(401, "Unauthorized");
} else {
    xlog("Credentials verified — saving registration");
}

## 2. Proxy Authentication for INVITE

Pattern requiring Proxy authentication (407) for call requests.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=inv1
To: <sip:1002@example.com>

In [ ]:
# Auth state simulation
$var(authenticated) = "no";

if ($var(authenticated) != "yes") {
    xlog("No Proxy-Authorization header — sending 407");
    send_reply(407, "Proxy Authentication Required");
} else {
    xlog("Authenticated — proceeding with routing");
}

## 3. Digest Authentication Flow Simulation

Real authentication requires DB lookups, but here we simulate the flow.

Actual Kamailio code:
```kamailio
# Production code
if (!www_authorize("example.com", "subscriber")) {
    www_challenge("example.com", "1");
    exit;
}
```

In [ ]:
# First attempt: no auth
%%sip INVITE
From: <sip:1001@example.com>;tag=first
To: <sip:2001@example.com>

In [ ]:
# Auth check — no header, respond with 407
xlog("Step 1: UA sends INVITE without credentials");
send_reply(407, "Proxy Authentication Required");
xlog("Step 2: Proxy responds with 407 + challenge");

In [ ]:
# Second attempt: UA resends with credentials
%%sip INVITE
From: <sip:1001@example.com>;tag=first
To: <sip:2001@example.com>

In [ ]:
# Auth success simulation
xlog("Step 3: UA resends INVITE with credentials");
$var(auth_ok) = "yes";

if ($var(auth_ok) == "yes") {
    xlog("Step 4: Authentication successful");
    xlog("Proceeding to route call to $ru");
} else {
    send_reply(403, "Forbidden");
}

## 4. Handling Authentication Failures

Handling wrong passwords or unknown accounts.

Production code:
```kamailio
if (!proxy_authorize("", "subscriber")) {
    proxy_challenge("", "1");
    exit;
}
if (!db_check_from()) {
    sl_send_reply("403", "Forbidden");
    exit;
}
consume_credentials();  # Security: remove auth headers
```

In [ ]:
# Wrong credentials simulation
$var(auth_result) = "fail";

if ($var(auth_result) == "fail") {
    xlog("Authentication FAILED — wrong password or unknown user");
    send_reply(403, "Forbidden");
} else {
    xlog("Authentication OK");
}

## 5. Per-Method Authentication Policy

In practice, different methods have different authentication policies:

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=p1
To: <sip:2001@example.com>

In [ ]:
# Per-method auth policy simulation
$var(authenticated) = "yes";

if (is_method("REGISTER")) {
    xlog("REGISTER: requires www_authorize");
    if ($var(authenticated) != "yes") {
        send_reply(401, "Unauthorized");
    } else {
        xlog("REGISTER authenticated — saving location");
    }
} else {
    if (is_method("INVITE")) {
        xlog("INVITE: requires proxy_authorize");
        if ($var(authenticated) != "yes") {
            send_reply(407, "Proxy Authentication Required");
        } else {
            xlog("INVITE authenticated — routing call");
        }
    } else {
        xlog("Other method: no auth required");
    }
}

## 6. consume_credentials() — Security Best Practice

After successful auth, `consume_credentials()` removes Authorization headers.
This prevents credentials from leaking to the next hop.

```kamailio
proxy_authorize("", "subscriber");
consume_credentials();  # Remove Authorization header
t_relay();              # Safe to relay
```

In [ ]:
# Post-auth header cleanup simulation
$var(user) = $(fu{uri.user});
xlog("Authenticated user: $var(user)");
xlog("Consuming credentials — removing auth headers");
xlog("Safe to relay: $ru");

---

### Summary

| Concept | Description |
|---------|-------------|
| WWW Auth (401) | Registrar → UA, used for REGISTER |
| Proxy Auth (407) | Proxy → UA, used for INVITE etc. |
| www_authorize | Verify credentials from DB |
| proxy_authorize | Verify proxy auth |
| consume_credentials | Remove auth headers (security) |
| Auth failure | 401/403/407 response |

Next notebook: **Intermediate/04-registration-and-location.ipynb** →